# Rata-rata bergerak
        **timeseries › forecasting** · notebook `moving_average`

        **Kapan dipakai:** Seri datar berfluktuasi di sekitar rata-rata (tanpa tren).

        **Jebakan umum:** Terlambat mengikuti tren; datar ke depan.

**Data demo:** CO2 bulanan (Mauna Loa). Split SELALU kronologis; evaluasi memakai holdout + rolling-origin.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
PATH      = ROOT / "data_latihan/timeseries/forecasting/co2_monthly.csv"
DATE_COL  = "date"; VALUE_COL = "value"
FREQ      = "MS"                 # MS=bulanan, QS=kuartalan, D=harian, H=jam, W=mingguan
M         = 12                  # panjang siklus musiman (12 bulanan, 4 kuartalan, 7 harian, 24 per jam)
H         = 36                 # horizon ramalan
N_ORIGINS, STEP = 3, 12   # rolling-origin: jumlah titik asal dan jaraknya
LAGS      = [1, 2, 3, 6, 12, 24]
EXOG_COLS = None                     # kolom variabel eksogen (None bila tidak ada)
WINDOW, EPOCHS = 2 * M, 150               # hanya untuk model neural
MODEL_KEY = "moving_average"
OUT = out_dir("timeseries", "forecasting", MODEL_KEY)
set_seed(42)
from superdb.timeseries import load_series, forecast_metrics, rolling_origin, plot_forecast, future_index

## 2. Baca & lihat data

In [ ]:
raw = pd.read_csv(PATH, parse_dates=[DATE_COL]).sort_values(DATE_COL)
y = load_series(PATH, DATE_COL, VALUE_COL, FREQ)
EX = raw.set_index(DATE_COL)[EXOG_COLS].astype(float).asfreq(FREQ).interpolate() if EXOG_COLS else None
print(f"{len(y)} titik: {y.index[0].date()} s/d {y.index[-1].date()} | freq={FREQ} | musiman M={M} | horizon H={H}")
ax = y.plot(figsize=(10, 3), title="Data"); plt.tight_layout(); plt.show()

## 3. Split kronologis
Test = `H` titik TERAKHIR. Tidak ada pengacakan.

In [ ]:
train, test = y.iloc[:-H], y.iloc[-H:]
print("latih:", len(train), "| uji:", len(test))

## 4. Model
Semua model berbentuk fungsi `fit_predict(train, h)` yang mengembalikan array ramalan sepanjang `h`. Kodenya bisa disalin.

In [ ]:
def fit_predict(train, h):
    return np.repeat(train.iloc[-M:].mean(), h)

In [ ]:
call = fit_predict
def seasonal_naive(tr, h): return np.tile(tr.iloc[-M:].to_numpy(), h // M + 1)[:h]
def naive(tr, h): return np.repeat(tr.iloc[-1], h)

## 5. Evaluasi holdout vs baseline
**MASE** = galat dibagi galat naive musiman 1 siklus pada data latih (skala tanpa satuan; < 1 berarti rata-rata galat ramalan lebih kecil dari galat naive 1 siklus di data latih). Untuk menilai manfaat model, **bandingkan langsung dengan baris baseline** di tabel: model yang tidak mengalahkan `seasonal_naive` pada horizon yang sama tidak layak dipakai.

In [ ]:
import time
t0 = time.time(); pred = np.asarray(call(train, H), float); fit_sec = time.time() - t0
assert pred.shape == (H,) and np.isfinite(pred).all(), "ramalan harus berisi H angka valid"
res = {MODEL_KEY: forecast_metrics(test.to_numpy(), pred, train.to_numpy(), M),
       "seasonal_naive (baseline)": forecast_metrics(test.to_numpy(), seasonal_naive(train, H), train.to_numpy(), M),
       "naive (baseline)": forecast_metrics(test.to_numpy(), naive(train, H), train.to_numpy(), M)}
cmp = pd.DataFrame(res).T; display(cmp.round(4)); print(f"waktu fit+ramal: {fit_sec:.1f} dtk")
plot_forecast(train, test, pred, title=f"{MODEL_KEY}: ramalan vs aktual", last=5 * M, path=OUT / "forecast.png"); plt.show()

## 6. Rolling-origin (lebih jujur daripada satu holdout)
Latih sampai beberapa titik asal berbeda dan rata-ratakan galatnya.

In [ ]:
ro, ro_mean = rolling_origin(y, call, H, N_ORIGINS, STEP, M)
display(ro.round(4)); print("rata-rata rolling-origin:", {k: round(v, 4) for k, v in ro_mean.items()})
sn, sn_mean = rolling_origin(y, lambda tr, h: seasonal_naive(tr, h), H, N_ORIGINS, STEP, M)
print(f"MASE rata-rata: model {ro_mean['mase']:.3f} vs seasonal naive {sn_mean['mase']:.3f}")

## 7. Ramalan ke depan (setelah data terakhir) → file submission
Model dilatih ulang pada SEMUA data lalu meramal `H` titik setelahnya.

In [ ]:
future = np.asarray(call(y, H), float)
sub = pd.DataFrame({"tanggal": future_index(y, H), "ramalan": future}); sub.to_csv(OUT / "submission.csv", index=False)
display(sub.head(3)); display(sub.tail(2))
plot_forecast(y, pd.Series(future, index=sub["tanggal"]), future, title="Ramalan ke depan", last=5 * M, path=OUT / "forecast_future.png"); plt.show()
save_json({"holdout": res[MODEL_KEY], "baseline_seasonal_naive": res["seasonal_naive (baseline)"], "rolling_origin_mean": ro_mean, "fit_seconds": fit_sec}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan model berdampingan lewat `MASE` rolling-origin. Statistik klasik (ETS/SARIMA/Theta) sering mengalahkan ML pada seri pendek.
- Gabungkan beberapa ramalan (rata-rata) hampir selalu menurunkan galat.
- Jika metrik lomba bukan MAE/RMSE (mis. sMAPE, RMSLE), pilih model dan transformasi (log) sesuai metrik itu.